# 02 — Data exploration

Explore one WikiANN configuration and preview the **lexical overlap analysis**
between a source training set and a target test set. All logic comes from
`src/axner` (`axner.data.stats`, `axner.analysis.lexical_overlap`).

Run notebook 01 (or the audit script) first so the data is cached.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))
print("repo root:", repo_root)

In [ ]:
from axner.constants import LABEL_NAMES
from axner.data.wikiann import load_wikiann

cache_dir = str(repo_root / "data" / "hf_cache")
jv_train = load_wikiann("jv", "train", cache_dir=cache_dir)
print(jv_train.num_rows, "rows; features:", list(jv_train.features))

In [ ]:
from axner.data.stats import dataset_statistics

dataset_statistics(jv_train, LABEL_NAMES, "jv_train")

In [ ]:
# Look at a few annotated examples.
def show_row(row):
    tags = [LABEL_NAMES[t] for t in row["ner_tags"]]
    return list(zip(row["tokens"], tags))

show_row(next(r for r in jv_train if any(t != 0 for t in r["ner_tags"])))

In [ ]:
import pandas as pd

lengths = pd.Series([len(r["tokens"]) for r in jv_train])
lengths.describe().to_frame("tokens_per_sentence")

In [ ]:
try:
    import matplotlib.pyplot as plt

    lengths.plot.hist(bins=20, title="jv train sentence lengths")
    plt.show()
except ImportError:
    print("matplotlib not installed - skipping plot (pip install matplotlib)")

## Lexical overlap preview (id train vs jv test)

Mentions from the target **test** set are classified against the source
**training** inventory: `seen` (exact normalized surface+type match),
`similar` (char n-gram Dice >= threshold), `unseen`. This is a post-training
analysis; test annotations are never used for training or checkpoint
selection.

In [ ]:
from axner.analysis.lexical_overlap import overlap_report

id_train = load_wikiann("id", "train", cache_dir=cache_dir)
jv_test = load_wikiann("jv", "test", cache_dir=cache_dir)

overlap_report(jv_test, id_train, LABEL_NAMES, threshold=0.5)

In [ ]:
# Threshold sweep - how sensitive is the seen/similar split?
for threshold in (0.3, 0.5, 0.7):
    rep = overlap_report(jv_test, id_train, LABEL_NAMES, threshold=threshold)
    print(threshold, {k: v for k, v in rep["mentions"].items()})

**Normalization (fixed, documented in `docs/methodology.md`):** Unicode NFC,
casefold, whitespace collapse. **Similarity:** Dice over character 2-4-grams
of the normalized surface with `^...$` boundary markers, entity-type-aware.